# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damiannogueira/flyrank-ml-intership/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — The Freshness Multiplier

The paper reports that the 31–90 day freshness window is the strongest stable measured growth band, with a growth-to-decline ratio of 7.88:1. It also reports that 365+ day content refreshed within 30 days shows substantially higher health and impressions than older content that was not recently refreshed.

The underlying growing/declining distinction comes from the paper's trend-direction definition, which is based on 30-day impression change versus the previous 30 days.

**Methodology question:**  
Are refreshed and non-refreshed pages sufficiently comparable before the refresh occurs?

Pages selected for refresh may already have stronger historical visibility, demand, or strategic importance. If so, part of the observed difference could reflect selection effects rather than the refresh itself. I would want to know whether the comparison controls for prior performance or uses matched cohorts before interpreting the observed association as evidence of refresh impact.

---

### Finding 2 — AI-Generated Content Is Not Penalized by Default

The paper reports that age-controlled AI-model cohorts do not show a simple blanket performance penalty tied only to AI use. Outcomes vary across model cohorts, and the paper interprets differences as more consistent with process quality, editing, topic fit, and rollout timing than with a universal AI penalty.

**Methodology question:**  
Is age control alone sufficient to support the claim that there is no blanket penalty associated with AI-generated content?

Age is an important confounder, but topic mix, search intent, competition, editing quality, client differences, and publishing strategy may also affect the observed outcomes. I would want to know whether these factors are balanced or controlled before interpreting the cohort differences as evidence against a general AI-related penalty.

These questions are not meant to reject the findings. They identify what additional validation would make the interpretations stronger and more transferable.

In [1]:
# Summarize the two paper findings selected for methodology review.
# This cell does not test the paper; it documents what we will audit.

import pandas as pd

paper_audit = pd.DataFrame(
    [
        {
            "finding": "Freshness Multiplier",
            "observed_evidence": "31-90d growth-to-decline ratio = 7.88:1",
            "methodology_focus": "Selection bias and pre-refresh comparability",
        },
        {
            "finding": "AI-generated content is not penalized by default",
            "observed_evidence": "Age-controlled cohorts show mixed model outcomes",
            "methodology_focus": "Whether age control is sufficient for confounding",
        },
    ]
)

paper_audit


,finding,observed_evidence,methodology_focus
0,Freshness Multiplier,31-90d growth-to-decline ratio = 7.88:1,Selection bias and pre-refresh comparability
1,AI-generated content is not penalized by default,Age-controlled cohorts show mixed model outcomes,Whether age control is sufficient for confounding


## 2. My model under an honest split (before/after)

### Before/after validation design

I will re-run the Week-5 Logistic Regression model under two validation designs while keeping the feature set, target, metric, and random seed fixed.

**Before:** a row-level random split. This allows pages from the same client to appear in both train and test data.

**After:** a client-grouped split. All pages belonging to the same `client_hash_id` stay entirely on one side of the split.

The grouped version is more aligned with the question of whether the model can generalize to clients it has not seen during training.

Both versions use the same pre-decision features from March 1–21, 2026 and the same later outcome from March 22–31, 2026. The evaluation metric remains Precision@50, and the positive base rate is reported alongside each result.

The comparison is intended to measure the effect of the validation design, not to change the model until a better score appears.

In [2]:
# Rebuild the Week-5 modeling frame and compare
# random row-level validation with client-grouped validation.

%pip -q install duckdb scikit-learn

from google.colab import userdata
import duckdb
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# Fixed experiment settings from Week 5.
FEATURES = [
    "impressions_21d",
    "clicks_21d",
    "ctr_21d",
    "avg_position_21d",
    "active_days_21d",
]

TARGET = "future_decline_target"
K = 50
RANDOM_STATE = 42


# Load the Hugging Face token securely from Colab Secrets.
HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)


# March 2026 remains the mid-panel development month.
MARCH_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""


# Build one page-level row:
# features from March 1-21,
# outcome from March 22-31.
model_frame = con.sql(f"""
    WITH pre_decision AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS impressions_21d,
            SUM(gsc_clicks) AS clicks_21d,

            SUM(gsc_sum_position)
                / NULLIF(SUM(gsc_impressions), 0)
                AS avg_position_21d,

            COUNT(
                DISTINCT CASE
                    WHEN gsc_impressions > 0 THEN report_date
                END
            ) AS active_days_21d

        FROM {MARCH_REL}

        WHERE
            gsc_data_available IS TRUE
            AND report_date
                BETWEEN DATE '2026-03-01'
                AND DATE '2026-03-21'

        GROUP BY
            client_hash_id,
            content_hash_id

        HAVING SUM(gsc_impressions) > 0
    ),

    later_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS future_impressions_10d

        FROM {MARCH_REL}

        WHERE
            gsc_data_available IS TRUE
            AND report_date
                BETWEEN DATE '2026-03-22'
                AND DATE '2026-03-31'

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        p.client_hash_id,
        p.content_hash_id,
        p.impressions_21d,
        p.clicks_21d,
        p.avg_position_21d,
        p.active_days_21d,
        f.future_impressions_10d

    FROM pre_decision AS p

    INNER JOIN later_window AS f
        USING (client_hash_id, content_hash_id)
""").df()


# CTR = clicks divided by impressions, expressed as a percentage.
model_frame["ctr_21d"] = (
    100.0
    * model_frame["clicks_21d"]
    / model_frame["impressions_21d"]
)


# Compare daily impression rates because the two windows
# have different lengths: 21 days versus 10 days.
model_frame["future_change_pct"] = (
    (
        model_frame["future_impressions_10d"] / 10
        - model_frame["impressions_21d"] / 21
    )
    / (model_frame["impressions_21d"] / 21)
    * 100
)


# Positive label = observed future daily-impression decline > 20%.
model_frame[TARGET] = (
    model_frame["future_change_pct"] < -20
).astype(int)


# Stable row order for reproducibility.
model_frame = model_frame.sort_values(
    ["client_hash_id", "content_hash_id"]
).reset_index(drop=True)


def build_model():
    """Create the same Logistic Regression pipeline used in Week 5."""
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "logistic_regression",
                LogisticRegression(
                    max_iter=1000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def precision_at_k(scores, labels, ids, k=50):
    """Rank scores deterministically and return Precision@K."""
    ranking = pd.DataFrame(
        {
            "score": scores,
            "label": labels.to_numpy(),
            "content_hash_id": ids.to_numpy(),
        }
    )

    ranking = ranking.sort_values(
        ["score", "content_hash_id"],
        ascending=[False, True],
        kind="stable",
    )

    return ranking.head(k)["label"].mean()


# ---------------------------------------------------------
# BEFORE: random row-level split
# ---------------------------------------------------------

random_train, random_test = train_test_split(
    model_frame,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=model_frame[TARGET],
)

random_model = build_model()

random_model.fit(
    random_train[FEATURES],
    random_train[TARGET],
)

random_scores = random_model.predict_proba(
    random_test[FEATURES]
)[:, 1]

random_p50 = precision_at_k(
    random_scores,
    random_test[TARGET],
    random_test["content_hash_id"],
    k=K,
)

random_base_rate = random_test[TARGET].mean()

random_client_overlap = len(
    set(random_train["client_hash_id"]).intersection(
        set(random_test["client_hash_id"])
    )
)


# ---------------------------------------------------------
# AFTER: client-grouped split
# ---------------------------------------------------------

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

group_train_idx, group_test_idx = next(
    group_splitter.split(
        model_frame,
        model_frame[TARGET],
        groups=model_frame["client_hash_id"],
    )
)

group_train = model_frame.iloc[group_train_idx].copy()
group_test = model_frame.iloc[group_test_idx].copy()

group_model = build_model()

group_model.fit(
    group_train[FEATURES],
    group_train[TARGET],
)

group_scores = group_model.predict_proba(
    group_test[FEATURES]
)[:, 1]

group_p50 = precision_at_k(
    group_scores,
    group_test[TARGET],
    group_test["content_hash_id"],
    k=K,
)

group_base_rate = group_test[TARGET].mean()

group_client_overlap = len(
    set(group_train["client_hash_id"]).intersection(
        set(group_test["client_hash_id"])
    )
)


# Show the before/after validation comparison.
split_comparison = pd.DataFrame(
    {
        "validation_design": [
            "Random row split",
            "Client-grouped split",
        ],
        "test_rows": [
            len(random_test),
            len(group_test),
        ],
        "client_overlap": [
            random_client_overlap,
            group_client_overlap,
        ],
        "base_rate": [
            random_base_rate,
            group_base_rate,
        ],
        "precision_at_50": [
            random_p50,
            group_p50,
        ],
    }
)

split_comparison["base_rate"] = (
    split_comparison["base_rate"] * 100
).round(1)

split_comparison["precision_at_50"] = (
    split_comparison["precision_at_50"] * 100
).round(1)

split_comparison


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,validation_design,test_rows,client_overlap,base_rate,precision_at_50
0,Random row split,29019,40,29.9,46.0
1,Client-grouped split,8852,0,29.9,28.0


### Validation comparison

The validation design materially changes the measured performance.

With a random row-level split, Logistic Regression reaches **46.0% Precision@50**, compared with a **29.9%** positive base rate. However, this split has **40 clients appearing in both training and test data**.

Under the client-grouped split, client overlap falls to **0**, and Precision@50 decreases to **28.0%** while the test base rate remains **29.9%**.

The 18-point drop suggests that the random split produced an overly optimistic estimate for the question I actually care about: whether the model can rank pages for clients it has not seen during training.

The grouped result is therefore the more defensible validation number. Under this design, the model does not provide measured lift over the held-out base rate at Precision@50.

This does not show that the random result was invalid as a calculation. It shows that it answered an easier question because train and test contained pages from many of the same clients.

## 3. Leakage audit

### Leakage audit

The prediction point is the end of March 21, 2026. Therefore, every model feature must be fully knowable using information available on or before that date.

The final Week-5 feature set passes this timeline check:

- `impressions_21d`: March 1–21 search impressions.
- `clicks_21d`: March 1–21 search clicks.
- `ctr_21d`: derived only from pre-decision clicks and impressions.
- `avg_position_21d`: weighted search position from March 1–21.
- `active_days_21d`: number of pre-decision days with search impressions.

None of these features is derived from the future outcome or from an existing FlyRank optimization flag.

By contrast, `future_impressions_10d`, `future_change_pct`, and `future_decline_target` belong to the outcome side of the timeline and must never be model inputs.

To verify that the evaluation harness can detect leakage, I will deliberately add `future_change_pct` as a temporary suspect feature. This variable directly contributes to the target definition, so a large performance jump would confirm that the audit is sensitive to label-derived leakage. The leaky result is diagnostic only and will not be kept as a valid model result.

In [3]:
# Audit the final features for timeline and label leakage.
# Then deliberately add one known leaky feature to verify
# that the validation setup is capable of detecting leakage.

feature_audit = pd.DataFrame(
    [
        {
            "feature": "impressions_21d",
            "source_window": "Mar 1-21",
            "known_at_prediction_time": True,
            "label_derived": False,
            "product_flag": False,
            "status": "SAFE",
        },
        {
            "feature": "clicks_21d",
            "source_window": "Mar 1-21",
            "known_at_prediction_time": True,
            "label_derived": False,
            "product_flag": False,
            "status": "SAFE",
        },
        {
            "feature": "ctr_21d",
            "source_window": "Mar 1-21",
            "known_at_prediction_time": True,
            "label_derived": False,
            "product_flag": False,
            "status": "SAFE",
        },
        {
            "feature": "avg_position_21d",
            "source_window": "Mar 1-21",
            "known_at_prediction_time": True,
            "label_derived": False,
            "product_flag": False,
            "status": "SAFE",
        },
        {
            "feature": "active_days_21d",
            "source_window": "Mar 1-21",
            "known_at_prediction_time": True,
            "label_derived": False,
            "product_flag": False,
            "status": "SAFE",
        },
    ]
)

print("Final feature audit:")
display(feature_audit)


# ---------------------------------------------------------
# Deliberate leakage test
# ---------------------------------------------------------

LEAKY_FEATURES = FEATURES + ["future_change_pct"]

leaky_model = build_model()

leaky_model.fit(
    group_train[LEAKY_FEATURES],
    group_train[TARGET],
)

leaky_scores = leaky_model.predict_proba(
    group_test[LEAKY_FEATURES]
)[:, 1]

leaky_p50 = precision_at_k(
    leaky_scores,
    group_test[TARGET],
    group_test["content_hash_id"],
    k=K,
)


leakage_test = pd.DataFrame(
    {
        "model_version": [
            "Honest grouped model",
            "Grouped model + future_change_pct",
        ],
        "precision_at_50": [
            group_p50,
            leaky_p50,
        ],
        "base_rate": [
            group_base_rate,
            group_base_rate,
        ],
    }
)

leakage_test["precision_at_50"] = (
    leakage_test["precision_at_50"] * 100
).round(1)

leakage_test["base_rate"] = (
    leakage_test["base_rate"] * 100
).round(1)

print()
print("Leakage sensitivity test:")
display(leakage_test)


# ---------------------------------------------------------
# Real failure examples from the honest grouped model
# ---------------------------------------------------------

error_frame = group_test[
    [
        "client_hash_id",
        "content_hash_id",
        TARGET,
        "impressions_21d",
        "clicks_21d",
        "ctr_21d",
        "avg_position_21d",
        "active_days_21d",
    ]
].copy()

error_frame["model_score"] = group_scores

error_frame = error_frame.sort_values(
    ["model_score", "client_hash_id", "content_hash_id"],
    ascending=[False, True, True],
    kind="stable",
).reset_index(drop=True)

error_frame.insert(
    0,
    "model_rank",
    range(1, len(error_frame) + 1),
)


# False positives = highly ranked pages whose observed target is 0.
top50_errors = error_frame.head(K)

false_positives = top50_errors[
    top50_errors[TARGET] == 0
].copy()

print()
print(f"False positives in grouped top {K}: {len(false_positives)}")

failure_examples = false_positives.head(3)[
    [
        "model_rank",
        "content_hash_id",
        "model_score",
        "impressions_21d",
        "clicks_21d",
        "ctr_21d",
        "avg_position_21d",
        "active_days_21d",
        TARGET,
    ]
]

failure_examples


Final feature audit:


,feature,source_window,known_at_prediction_time,label_derived,product_flag,status
0,impressions_21d,Mar 1-21,True,False,False,SAFE
1,clicks_21d,Mar 1-21,True,False,False,SAFE
2,ctr_21d,Mar 1-21,True,False,False,SAFE
3,avg_position_21d,Mar 1-21,True,False,False,SAFE
4,active_days_21d,Mar 1-21,True,False,False,SAFE



Leakage sensitivity test:


,model_version,precision_at_50,base_rate
0,Honest grouped model,28.0,29.9
1,Grouped model + future_change_pct,100.0,29.9



False positives in grouped top 50: 36


,model_rank,content_hash_id,model_score,impressions_21d,clicks_21d,ctr_21d,avg_position_21d,active_days_21d,future_decline_target
0,1,content_1bc14f25ea5989e0,0.434111,11820.0,4.0,0.033841,8.110152,21,0
1,2,content_31ea2c45cdc6c46e,0.408457,5482.0,1.0,0.018242,10.909887,21,0
3,4,content_fbe9d7902c683443,0.398929,4175.0,1.0,0.023952,7.157126,21,0


### Leakage audit result

The final Week-5 feature set passes the timeline and label-leakage audit. All five model inputs are computed only from the March 1–21 pre-decision window, none is derived from the future target, and none is an existing FlyRank workflow flag.

The deliberate leakage test confirms that the validation setup is sensitive to answer leakage. When `future_change_pct` is added as a temporary feature, grouped Precision@50 jumps from **28.0% to 100.0%**.

This is expected because `future_change_pct` directly contributes to the definition of `future_decline_target`. The leaky score is therefore not a valid performance result; it is evidence that the audit can detect a feature that contains outcome information.

The honest grouped model still produces **36 false positives in its top 50**. The inspected errors again show pages with sustained visibility, very few clicks, and extremely low CTR. These examples reinforce that the observed pre-decision pattern is not sufficient to identify future decline reliably.

The final model therefore keeps only the five pre-decision features and excludes all future-window, label-derived, and product-decision variables.

## 4. Claim rewrite

**Earlier claim:**

> Logistic Regression improves on the Week-4 rule.

That statement is directionally true for the specific grouped evaluation used in Week 5, but it can sound broader than the evidence supports.

**Safer claim:**

> On the held-out client-grouped evaluation used in this study, Logistic Regression achieved **28.0% Precision@50**, compared with **20.0%** for the Week-4 rule baseline. However, the model did not exceed the **29.9%** positive base rate, and its measured performance was substantially lower than under a random row-level split. The result therefore supports the model only as an exploratory decision-support ranking, not as evidence of reliable generalization across unseen clients.

This wording keeps the observed improvement over the rule baseline while making the validation limitation and lack of lift over the grouped-test base rate explicit.

In [4]:
# Record the original and revised claim for the validation audit.

claim_audit = pd.DataFrame(
    [
        {
            "claim_version": "Earlier",
            "claim": "Logistic Regression improves on the Week-4 rule.",
            "risk": "Too broad without validation context",
        },
        {
            "claim_version": "Rewritten",
            "claim": (
                "On the held-out client-grouped evaluation, Logistic Regression "
                "reached 28.0% Precision@50 versus 20.0% for the Week-4 baseline, "
                "but did not exceed the 29.9% test base rate."
            ),
            "risk": "Public-safe and scoped to measured evidence",
        },
    ]
)

claim_audit


,claim_version,claim,risk
0,Earlier,Logistic Regression improves on the Week-4 rule.,Too broad without validation context
1,Rewritten,"On the held-out client-grouped evaluation, Log...",Public-safe and scoped to measured evidence


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.